# Gold Notebook: `student_360`


## Set up environment

In [2]:
import os
import sys
from pathlib import Path

for env_name in ("SPARK_HOME", "HADOOP_CONF_DIR", "SPARK_CONF_DIR"):
    os.environ.pop(env_name, None)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import (
    functions as F,
    Window
)


In [3]:
silver_root = project_root / "data" / "silver"
gold_root = project_root / "data" / "gold"

print("project_root:", project_root)
print("silver_root:", silver_root)
print("gold_root:", gold_root)


project_root: /mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation
silver_root: /mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/silver
gold_root: /mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/gold


In [4]:
spark = (
    SparkSession.builder
    .appName("learn-gold-student-360")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 01:27:13 WARN Utils: Your hostname, DESKTOP-8HR4EC1, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 01:27:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 01:27:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load Silver Inputs

Gold must consume trusted Silver only.


In [5]:
leads_silver_df = spark.read.parquet(str(silver_root / "leads"))
learning_silver_df = spark.read.parquet(str(silver_root / "learning_activity"))
sales_silver_df = spark.read.parquet(str(silver_root / "sales"))

print("leads_silver rows:", leads_silver_df.count())
print("learning_silver rows:", learning_silver_df.count())
print("sales_silver rows:", sales_silver_df.count())


leads_silver rows: 375
learning_silver rows: 929
sales_silver rows: 426


In [6]:
print(
    "leads duplicate student_code:",
    leads_silver_df.groupBy("student_code").count().filter(F.col("count") > 1).count(),
)

leads_silver_df.show(5, truncate=False)


leads duplicate student_code: 0
+------------+--------------+------------+------------------------+-----+-------------------+--------------+
|student_code|student_name  |mobile_phone|school_name             |grade|lead_created_at    |source_channel|
+------------+--------------+------------+------------------------+-----+-------------------+--------------+
|STU100000   |Petch Lertchai|0022768040  |Satriwithaya School     |M2   |2025-01-01 09:28:00|school_fair   |
|STU100001   |Jane Srisuk   |0577442871  |St. Gabriel's College   |M3   |2026-01-19 14:33:00|tiktok        |
|STU100002   |Arisa Rattan  |0930086875  |St. Gabriel's College   |M1   |2025-12-13 03:56:00|school_fair   |
|STU100003   |Leo Ong       |0831067988  |St. Gabriel's College   |M3   |2025-09-19 12:10:00|line          |
|STU100004   |Petch Khan    |0136760652  |Samsen Wittayalai School|M4   |2025-04-23 15:43:00|tiktok        |
+------------+--------------+------------+------------------------+-----+-------------------+---

## Engagement


In [7]:
learning_silver_df.orderBy("student_code","event_time").show(1000)

+------------+-------------------+---------------+--------------+------------+----------------+--------+
|student_code|         event_time|     event_name|       subject|     chapter|duration_minutes|platform|
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|   STU100000|2025-06-10 23:03:00|    quiz_submit|          Math|  Live Class|              11| android|
|   STU100000|2025-07-10 03:57:00|    video_watch|          Thai|       Intro|              45|     ios|
|   STU100000|2025-08-11 13:35:00|    lesson_open|     Chemistry|  Live Class|              62|  tablet|
|   STU100000|2025-09-18 22:42:00|    quiz_submit|Social Studies|   Chapter 1|              75| android|
|   STU100000|2025-11-22 05:15:00|    lesson_open|Social Studies|       Intro|               8| android|
|   STU100001|2025-06-12 18:54:00| download_sheet|       Biology|   Chapter 3|              14|  tablet|
|   STU100001|2025-09-18 12:23:00|    quiz_submit|Socia

In [8]:
reference_ts = learning_silver_df.select(F.max("event_time")).first()[0]
before_7d = reference_ts - timedelta(days=7)
before_30d = reference_ts - timedelta(days=30)

engagement_df = (
    learning_silver_df
    .groupBy("student_code")
    .agg(
        F.count("*").alias("total_learning_events"),
        F.sum("duration_minutes").alias("total_learning_minutes"),
        F.sum(
            F.when(
                (F.col("event_time") <= F.lit(reference_ts)) &
                (F.col("event_time") > F.lit(before_7d)),
                1
            ).otherwise(0)
        ).alias("events_last_7d"),
        F.sum(
            F.when(
                (F.col("event_time") <= F.lit(reference_ts)) &
                (F.col("event_time") > F.lit(before_30d)),
                1
            ).otherwise(0)
        ).alias("events_last_30d"),
        F.sum(
            F.when(F.col("event_name") == "quiz_submit", 1).otherwise(0)
        ).alias("quiz_submit_events")
    )
)

engagement_df.show(1000)

+------------+---------------------+----------------------+--------------+---------------+------------------+
|student_code|total_learning_events|total_learning_minutes|events_last_7d|events_last_30d|quiz_submit_events|
+------------+---------------------+----------------------+--------------+---------------+------------------+
|   STU100268|                    1|                    82|             0|              0|                 0|
|   STU100363|                    1|                    67|             0|              0|                 0|
|   STU100397|                    4|                   152|             0|              0|                 0|
|   STU100004|                    4|                   162|             0|              0|                 0|
|   STU100024|                    4|                   302|             0|              1|                 1|
|   STU100116|                    3|                   153|             0|              0|                 1|
|   STU100

In [19]:
print(f"First time = {learning_silver_df.select(F.min("event_time")).first()[0]}")
print(f"Latest time = {reference_ts}")

First time = 2025-06-01 08:11:00
Latest time = 2026-03-19 05:55:00


## Revenue

In [10]:
sales_silver_df.show(1000)

+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|student_code|    order_id|         order_date|        product_name|   product_type| amount|payment_status|
+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|   STU100407|ORD202500000|2025-10-01 12:15:00|Math Intensive Bo...| mock_exam_pack| 990.00|       success|
|   STU100103|ORD202500003|2025-12-10 14:14:00|M6 Final Review B...|recorded_course|1990.00|       success|
|   STU100169|ORD202500006|2025-12-29 00:01:00| TCAS Mock Exam Pack|   trial_course|   0.00|       success|
|   STU100066|ORD202500008|2025-07-10 04:47:00|    Math Trial Class|   trial_course|   0.00|       success|
|   STU100285|ORD202500011|2025-12-21 09:00:00|   Biology Exam Prep|    live_course| 490.00|       success|
|   STU100087|ORD202500014|2025-11-28 04:18:00|Physics Formula M...|   trial_course|   0.00|       success|
|   STU100412|ORD202500015|2

In [11]:
# trial_course will not be evaluated in revenue metrics
# so this dataframe will only have paying student
# the not paying student will have NULL on total_paid_orders, total_revenue, first_purchase_at, 
#                                          last_purchase_at, is_paying_student after join with leads

revenue_df = (
    sales_silver_df
    .filter(F.col("product_type") != F.lit("trial_course"))
    .groupBy(F.col("student_code"))
    .agg(
        F.count("*").alias("total_paid_orders"),
        F.sum(F.col("amount")).alias("total_revenue"),
        F.min(F.col("order_date")).alias("first_purchase_at"),
        F.max(F.col("order_date")).alias("last_purchase_at")
    )
)

revenue_df.show(1000)

+------------+-----------------+-------------+-------------------+-------------------+
|student_code|total_paid_orders|total_revenue|  first_purchase_at|   last_purchase_at|
+------------+-----------------+-------------+-------------------+-------------------+
|   STU100172|                2|      5480.00|2025-11-19 23:33:00|2026-02-04 16:40:00|
|   STU100417|                1|       790.00|2026-02-04 01:22:00|2026-02-04 01:22:00|
|   STU100064|                1|      2990.00|2025-12-14 23:49:00|2025-12-14 23:49:00|
|   STU100249|                1|      1990.00|2025-08-02 21:47:00|2025-08-02 21:47:00|
|   STU100264|                1|      4990.99|2025-07-09 10:24:00|2025-07-09 10:24:00|
|   STU100078|                1|       990.00|2025-08-09 12:33:00|2025-08-09 12:33:00|
|   STU100068|                1|       790.00|2025-07-05 02:55:00|2025-07-05 02:55:00|
|   STU100204|                1|      3990.00|2025-10-26 10:22:00|2025-10-26 10:22:00|
|   STU100261|                3|     13970.

## After join

In [12]:
df_gold_student_360_1 = (
    leads_silver_df.alias("l")
    .join(engagement_df, on="student_code", how="left")
    .join(revenue_df, on="student_code", how="left")
)

df_gold_student_360_1.show(1000)

+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+
|student_code|    student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|total_learning_events|total_learning_minutes|events_last_7d|events_last_30d|quiz_submit_events|total_paid_orders|total_revenue|  first_purchase_at|   last_purchase_at|
+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+
|   STU100000|  Petch Lertchai|  0022768040| Satriwithaya School|   M2|2025-01-01 09:28:00|   school_fair|                    5|                   201|             0|            

### Handling NULL on revenue after join

In [13]:
# the person who has no information will be classified as inactive so put 0 on these column
# keep first_purchase_at and last_purchase_at NULL because the trial course is not actual transaction

df_gold_student_360_2 = (
     df_gold_student_360_1
     .fillna({
         "total_learning_events": 0,
         "total_learning_minutes": 0,
         "events_last_7d": 0,
         "events_last_30d": 0,
         "quiz_submit_events": 0,
         "total_paid_orders": 0,
         "total_revenue": 0
     })
)

df_gold_student_360_2.show(1000)

+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+
|student_code|    student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|total_learning_events|total_learning_minutes|events_last_7d|events_last_30d|quiz_submit_events|total_paid_orders|total_revenue|  first_purchase_at|   last_purchase_at|
+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+
|   STU100000|  Petch Lertchai|  0022768040| Satriwithaya School|   M2|2025-01-01 09:28:00|   school_fair|                    5|                   201|             0|            

In [14]:
df_gold_student_360_2.printSchema()

root
 |-- student_code: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- mobile_phone: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- lead_created_at: timestamp (nullable = true)
 |-- source_channel: string (nullable = true)
 |-- total_learning_events: long (nullable = false)
 |-- total_learning_minutes: long (nullable = false)
 |-- events_last_7d: long (nullable = false)
 |-- events_last_30d: long (nullable = false)
 |-- quiz_submit_events: long (nullable = false)
 |-- total_paid_orders: long (nullable = false)
 |-- total_revenue: decimal(22,2) (nullable = false)
 |-- first_purchase_at: timestamp (nullable = true)
 |-- last_purchase_at: timestamp (nullable = true)



## Engage Segments

In [15]:
df_gold_student_360_3 = (
    df_gold_student_360_2
    .withColumn(
        "engagement_segment",
        F.when(
             (F.col("events_last_7d") >= F.lit(5)) &
             (F.col("events_last_30d") >= F.lit(12)),
             F.lit("high")
         )
         .when(
             (F.col("events_last_7d") >= F.lit(2)) &
             (F.col("events_last_30d") >= F.lit(5)),
             F.lit("medium")
         )
         .when(
             (F.col("events_last_7d") >= F.lit(0)) &
             (F.col("events_last_30d") > F.lit(0)),
             F.lit("low")
         )
         .when(F.col("events_last_30d") == F.lit(0), F.lit("inactive"))
    )
)

df_gold_student_360_3.show(1000)

+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+------------------+
|student_code|    student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|total_learning_events|total_learning_minutes|events_last_7d|events_last_30d|quiz_submit_events|total_paid_orders|total_revenue|  first_purchase_at|   last_purchase_at|engagement_segment|
+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+------------------+
|   STU100000|  Petch Lertchai|  0022768040| Satriwithaya School|   M2|2025-01-01 09:28:00|   school_fair|               

## Flags

### is_paying_student

In [18]:
df_gold_student_360_4 = (
    df_gold_student_360_3
    .withColumn(
        "is_paying_student", F.when(
            F.col("total_paid_orders") > 0, F.lit(True)
        ).otherwise(F.lit(False))
    )
)

df_gold_student_360_4.show(1000)

+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+------------------+-----------------+
|student_code|    student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|total_learning_events|total_learning_minutes|events_last_7d|events_last_30d|quiz_submit_events|total_paid_orders|total_revenue|  first_purchase_at|   last_purchase_at|engagement_segment|is_paying_student|
+------------+----------------+------------+--------------------+-----+-------------------+--------------+---------------------+----------------------+--------------+---------------+------------------+-----------------+-------------+-------------------+-------------------+------------------+-----------------+
|   STU100000|  Petch Lertchai|  0022768040| Satriwithaya School|  

### risk_flag

### lead_priority_score

### risk_priority_band